## File Import

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output

In [2]:
file=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\FBG_Data\FBG_SKU_Attributes.csv"

In [3]:
df=pd.read_csv(file, encoding='utf-8', low_memory=False, keep_default_na=False, na_values=[''])
# df=df.drop_duplicates()
# df = df[df['PartTerminologyName'].notna()]
# df

In [4]:
df

,Product Group,Source,BrandName,Brand,PartNumber,PartTerminologyName,Status,PAName,Attribute_Type,Value,AC_AttributeCount,Attribute_Value,Key,Parent-Child- Customer Brand
0,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Casting Number,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpCasting Number,Parent
1,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Fan Clutch Included,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpFan Clutch Included,Parent
2,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Grade Type,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpGrade Type,Parent
3,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Housing Material,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpHousing Material,Parent
4,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Hub Height,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpHub Height,Parent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11428054,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Lens Width,FBG_Attribute,5.5,0.0,0.0,8261605Agriculture LightLens Width,Child
11428055,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Prop 65,FBG_Attribute,Yes,0.0,0.0,8261605Agriculture LightProp 65,Child
11428056,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Prop 65 Chemical,FBG_Attribute,DEHP,0.0,0.0,8261605Agriculture LightProp 65 Chemical,Child
11428057,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Prop 65 Warning,FBG_Attribute,WARNING: Cancer and Reproductive Harm – www.p6...,0.0,0.0,8261605Agriculture LightProp 65 Warning,Child


In [5]:
df['FileName']=df.Brand+"_"+df.PartTerminologyName

In [6]:
df['Product Group'].unique()

array(['Repair', 'Brakes', 'Towing', 'Steering/ Electronics', 'Filter',
       'Lighting', 'Wipers'], dtype=object)

In [7]:
df = df[df['Parent-Child- Customer Brand'] == "Parent"]

In [8]:
output_location=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\FBG_Data\Phase-2\FBG_Circulation"

In [10]:
df['Product Group'].unique()

array(['Repair', 'Brakes', 'Steering/ Electronics', 'Filter', 'Towing',
       'Wipers'], dtype=object)

In [13]:
import re
#filtering only Required Product Group
Type="Brakes"
df_IN=df[df['Product Group']==Type]
Filenames=df_IN.FileName.unique()
length=len(Filenames)
for i in tqdm(range (length)):
    BrandName=df[(df.FileName==Filenames[i])]['BrandName'].mode()[0]
    print(i, Filenames[i])
    # and separating Autocare Attributes and FBG Attributes
    df1=df_IN[(df_IN.FileName==Filenames[i]) & (df_IN['Attribute_Type']=="Autocare_Attribute")]
    df1=df1.drop(columns=['FileName'])    
    df1=df1[['PartNumber','PartTerminologyName', 'PAName', 'Value']]
    grouped=df1.groupby(['PartNumber','PartTerminologyName', 'PAName'])['Value'].agg(lambda x: list(x) if len(x) > 1 else x.iloc[0])
    # Unstack to get pivot-style format
    pivot_df_AC = grouped.unstack(fill_value=None)
    df2=df_IN[(df_IN.FileName==Filenames[i]) & (df_IN['Attribute_Type']!="Autocare_Attribute")]
    df2=df2.drop(columns=['FileName'])
    df2=df2[['PartNumber','PartTerminologyName', 'PAName', 'Value']]
    grouped=df2.groupby(['PartNumber','PartTerminologyName', 'PAName'])['Value'].agg(lambda x: list(x) if len(x) > 1 else x.iloc[0])
    # Unstack to get pivot-style format
    pivot_df_FBG= grouped.unstack(fill_value=None)
    Filename=re.sub(r"[^a-zA-Z0-9\s_]", "", Filenames[i].split("_")[1])
    with pd.ExcelWriter(rf"{output_location}\{Type}\\"+BrandName+"_"+Filename.replace(" ","")+".xlsx") as writer:
        pivot_df_AC.to_excel(writer, sheet_name="Autocare", index=True, engine='openpyxl')
        pivot_df_FBG.to_excel(writer, sheet_name="FBG", index=True, engine='openpyxl')
    #print("File FBG_SKU_Attributes_"+Filenames[i]+".xlsx created successfully")
    clear_output(wait=True)

100%|██████████| 121/121 [05:10<00:00,  2.57s/it]


In [ ]:
df1

,Product Group,Source,BrandName,Brand,PartNumber,PartTerminologyName,Status,PAName,Attribute_Type,Value,AC_AttributeCount,Attribute_Value,Key,Parent-Child- Customer Brand


In [40]:
df_IN=df[df['Product Group']==Type]
len(df_IN.PartTerminologyName.unique())

73

In [ ]:
Brand_Filenames=df.Brand.unique()
print(len(Brand_Filenames))
for i in range (len(Brand_Filenames)):
    print(i)
    df1=df[df.Brand==Brand_Filenames[i]]
    df1=df1.drop(columns=['Brand'])
    ProductNames=df1.PartTerminologyName.unique()
    FileName=rf"{output_location}\FBG_SKU_Attributes_"+Brand_Filenames[i]+".xlsx"
    # df1=df1[['PartNumber', 'PAName', 'Value']]
        # Create Excel writer per region
    with pd.ExcelWriter(FileName, engine='xlsxwriter') as writer:
        for j in range(len(ProductNames)):
            df2=df1[df1.PartTerminologyName==ProductNames[j]]
            # df2=df2[['PartNumber', 'PAName', 'Value']]
            grouped=df2.groupby(['PartTerminologyName','PartNumber', 'PAName'])['Value'].agg(lambda x: list(x) if len(x) > 1 else x.iloc[0])
            # Unstack to get pivot-style format
            pivot_df = grouped.unstack(fill_value=None)
            print("Processing Product:", ProductNames[j])
            ProductNames[j]=ProductNames[j].replace('/', '_').replace(' ', "")
            if len(ProductNames[j]) > 31:
                ProductNames[j] = ProductNames[j][:31]
            pivot_df.to_excel(writer, sheet_name=str(ProductNames[j]), index=True)
    print("File FBG_SKU_Attributes_"+Brand_Filenames[i]+".xlsx created successfully")
    clear_output(wait=True)
        

Processing Product: Engine Water Pump
Processing Product: Engine Auxiliary Water Pump
Processing Product: Drive Motor Inverter Cooler Water Pump
Processing Product: Electric Engine Water Pump
Processing Product: Engine Water Pump Pulley Bolt Cover
Processing Product: Engine Water Pump Adapter
Processing Product: Accessory Drive Belt Idler Assembly
Processing Product: Engine Timing Belt Kit with Water Pump
File FBG_SKU_Attributes_ASC.xlsx created successfully
Processing Product: Trunk Lid Lift Support
Processing Product: Hood Lift Support
Processing Product: Liftgate Lift Support
Processing Product: Multi-Purpose Lift Support
Processing Product: Convertible Top Cover Lift Support
Processing Product: Seat Adjustment Strut
Processing Product: Door Lift Support
Processing Product: Back Glass Lift Support
Processing Product: Tailgate Lift Support
Processing Product: Cargo Van Access Panel Lift Support
Processing Product: Truck Bed Storage Box Lid Lift Support
Processing Product: Top Stowage